In [1]:
"""
msa_imd_ranks.py
================
Reads IMD 2025 LSOA data, aggregates to MSA level,
calculates decile distributions and share of LSOAs in
most deprived 10% nationally. Ranks all 20 MSAs.

Run from project root:
  python scripts/msa_imd_ranks.py
"""

import pandas as pd
import numpy as np
import os

BASE     = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"
IMD_DIR  = f"{BASE}/data/raw/imd"

# ── 1. Discover files ──────────────────────────────────────────────────────
print("── 1. Scanning IMD directory ────────────────────────────────────")
for f in os.listdir(IMD_DIR):
    size_kb = os.path.getsize(f"{IMD_DIR}/{f}") / 1024
    print(f"   {f:<70}  {size_kb:,.0f} KB")

# ── 2. Load LSOA-level IMD (File 1) ───────────────────────────────────────
print("\n── 2. Loading LSOA-level IMD data ───────────────────────────────")

# Try to find the right file — look for CSVs first
lsoa_file = None
lad_file  = None

for f in os.listdir(IMD_DIR):
    fl = f.lower()
    if fl.endswith(".csv") and ("imd" in fl or "indices" in fl or "file_1" in fl or "data" in fl):
        lsoa_file = f"{IMD_DIR}/{f}"
        print(f"   Candidate LSOA file: {f}")
    if "file_10" in fl or "lad_summ" in fl or "local_authority" in fl:
        lad_file = f"{IMD_DIR}/{f}"
        print(f"   Candidate LAD file:  {f}")

# Load LSOA file
print(f"\n   Loading: {lsoa_file}")
if lsoa_file and lsoa_file.endswith(".csv"):
    lsoa_raw = pd.read_csv(lsoa_file)
else:
    # Try Excel
    for f in os.listdir(IMD_DIR):
        if "imd" in f.lower() and f.endswith(".xlsx") and "file_1" in f.lower():
            lsoa_file = f"{IMD_DIR}/{f}"
    lsoa_raw = pd.read_excel(lsoa_file)

print(f"   Shape: {lsoa_raw.shape}")
print(f"   Columns: {list(lsoa_raw.columns)}")
print(f"\n   First 3 rows:")
print(lsoa_raw.head(3).to_string())

# ── 3. Load LAD summary (File 10) ─────────────────────────────────────────
print("\n── 3. Loading LAD summary (File 10) ─────────────────────────────")
if lad_file:
    print(f"   Loading: {lad_file}")
    if lad_file.endswith(".xlsx"):
        # Print sheet names first
        xl = pd.ExcelFile(lad_file)
        print(f"   Sheet names: {xl.sheet_names}")
        # Load IMD sheet
        lad_raw = pd.read_excel(lad_file, sheet_name=0)
        print(f"   Shape: {lad_raw.shape}")
        print(f"   Columns: {list(lad_raw.columns)}")
        print(f"\n   First 3 rows:")
        print(lad_raw.head(3).to_string())
    elif lad_file.endswith(".csv"):
        lad_raw = pd.read_csv(lad_file)
        print(f"   Shape: {lad_raw.shape}")
        print(f"   Columns: {list(lad_raw.columns)}")
        print(f"\n   First 3 rows:")
        print(lad_raw.head(3).to_string())
else:
    print("   ⚠  No LAD summary file found — check filenames")

── 1. Scanning IMD directory ────────────────────────────────────
   File_10_-_IoD2025_Local_Authority_District_Summaries__lower-tier__v2.xlsx  222 KB
   IMD_2025_data.csv                                                       1,904 KB

── 2. Loading LSOA-level IMD data ───────────────────────────────
   Candidate LAD file:  File_10_-_IoD2025_Local_Authority_District_Summaries__lower-tier__v2.xlsx
   Candidate LSOA file: IMD_2025_data.csv

   Loading: /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map/data/raw/imd/IMD_2025_data.csv
   Shape: (33755, 6)
   Columns: ['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA']

   First 3 rows:
  LSOA code (2021)     LSOA name (2021) Local Authority District code (2024)

In [2]:
"""
msa_imd_ranks.py
================
Reads IMD 2025 LSOA data, aggregates to MSA level,
calculates decile distributions and share of LSOAs in
most deprived deciles. Ranks all 20 MSAs.

Key metric: % of LSOAs in most deprived 10% nationally (decile 1)
This is more honest than average rank as it captures concentration
of deprivation rather than smoothing it into a single number.

Run from project root:
  python scripts/msa_imd_ranks.py
"""

import pandas as pd
import numpy as np
import os

BASE     = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"
IMD_DIR  = f"{BASE}/data/raw/imd"
OA_LOOKUP= f"{BASE}/data/raw/lookups/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv"
CA_LOOKUP= f"{BASE}/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv"

# ── 1. Load IMD LSOA data ─────────────────────────────────────────────────
print("── 1. Loading IMD 2025 LSOA data ────────────────────────────────")
imd = pd.read_csv(f"{IMD_DIR}/IMD_2025_data.csv")
imd = imd.rename(columns={
    "LSOA code (2021)":                                                                    "lsoa_code",
    "LSOA name (2021)":                                                                    "lsoa_name",
    "Local Authority District code (2024)":                                                "lad24_code",
    "Local Authority District name (2024)":                                                "lad24_name",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)":                "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA":  "imd_decile",
})
print(f"   LSOAs loaded: {len(imd):,}")
print(f"   Decile distribution (England):")
print(imd["imd_decile"].value_counts().sort_index().to_string())

# ── 2. Load File 10 LAD summary (IMD sheet) ───────────────────────────────
print("\n── 2. Loading File 10 LAD summary ───────────────────────────────")
lad_imd = pd.read_excel(
    f"{IMD_DIR}/File_10_-_IoD2025_Local_Authority_District_Summaries__lower-tier__v2.xlsx",
    sheet_name="IMD"
)
lad_imd = lad_imd.rename(columns={
    "Local Authority District code (2024)":              "lad24_code",
    "Local Authority District name (2024)":              "lad24_name",
    "IMD - Average rank ":                               "imd_avg_rank",
    "IMD - Rank of average rank ":                       "imd_rank_of_avg_rank",
    "IMD - Proportion of LSOAs in most deprived 10% nationally ": "imd_pct_decile1_file10",
})
print(f"   LADs in File 10: {len(lad_imd):,}")
print(f"   Columns available: {list(lad_imd.columns)}")

# ── 3. Build LSOA → LAD → CA lookup ──────────────────────────────────────
print("\n── 3. Building LSOA → MSA lookup ───────────────────────────────")

# The IMD data already has LAD24 code — use that directly
# We just need to map LAD24 → CA via the CA lookup
ca_lookup = pd.read_csv(CA_LOOKUP)
ca_lookup  = ca_lookup[["LAD25CD","CAUTH25CD","CAUTH25NM"]].rename(columns={
    "LAD25CD": "lad_code", "CAUTH25CD": "ca_code", "CAUTH25NM": "ca_name"
})

# North Yorkshire reorganisation fix — LAD24 code E06000065 should be in YNYCA
# Check if it maps correctly
ny_check = ca_lookup[ca_lookup["lad_code"] == "E06000065"]
print(f"   North Yorkshire in CA lookup: {ny_check[['lad_code','ca_name']].values.tolist()}")

# Join IMD to CA lookup on LAD code
# IMD uses LAD24 codes, CA lookup uses LAD25 codes — mostly identical
imd_ca = imd.merge(ca_lookup, left_on="lad24_code", right_on="lad_code", how="left")

matched   = imd_ca["ca_code"].notna().sum()
unmatched = imd_ca["ca_code"].isna().sum()
print(f"   LSOAs matched to a CA: {matched:,}")
print(f"   LSOAs outside any CA:  {unmatched:,}")

# ── 4. Define MSA boundaries including pipeline areas ─────────────────────
print("\n── 4. Adding pipeline MSA areas ─────────────────────────────────")

# Pipeline areas: assign via LAD24 code directly (same codes as LAD25 for these)
PIPELINE_LADS = {
    "Sussex & Brighton":     ["E07000061","E07000062","E07000063","E07000064","E07000065",
                               "E07000223","E07000224","E07000225","E07000226","E07000227",
                               "E07000228","E07000229","E06000043"],
    "Hampshire & Solent":    ["E07000084","E07000085","E07000086","E07000087","E07000088",
                               "E07000089","E07000090","E07000091","E07000092","E07000093",
                               "E07000094","E06000044","E06000045"],
    "Cheshire & Warrington": ["E06000049","E06000050","E06000007"],
    "Cumbria":               ["E06000063","E06000064"],
    "Greater Essex":         ["E07000066","E07000067","E07000068","E07000069","E07000070",
                               "E07000071","E07000072","E07000073","E07000074","E07000075",
                               "E07000076","E07000077","E06000033","E06000034"],
    "Norfolk & Suffolk":     ["E07000143","E07000144","E07000145","E07000146","E07000147",
                               "E07000148","E07000149","E07000200","E07000202","E07000203",
                               "E07000244","E07000245"],
}

PIPELINE_NAMES = {v: k for k, codes in PIPELINE_LADS.items() for v in codes}

# Assign pipeline CA names where ca_name is null
mask = imd_ca["ca_code"].isna() & imd_ca["lad24_code"].isin(PIPELINE_NAMES)
imd_ca.loc[mask, "ca_name"] = imd_ca.loc[mask, "lad24_code"].map(PIPELINE_NAMES)
imd_ca.loc[mask, "ca_code"] = "PIPELINE"

for name in PIPELINE_LADS:
    n = (imd_ca["ca_name"] == name).sum()
    print(f"   {name}: {n} LSOAs assigned")

# Final in-scope MSA list
IN_SCOPE_CAS = [
    "E47000001","E47000002","E47000003","E47000004","E47000006",
    "E47000007","E47000008","E47000009","E47000012","E47000013",
    "E47000014","E47000016","E47000017","E47000018","PIPELINE",
]

imd_inscope = imd_ca[
    imd_ca["ca_code"].isin(IN_SCOPE_CAS) |
    imd_ca["ca_name"].isin(list(PIPELINE_LADS.keys()))
].copy()

print(f"\n   In-scope LSOAs: {len(imd_inscope):,}")
print(f"   MSAs covered:   {imd_inscope['ca_name'].nunique()}")

# Use ca_name as the grouping key — standardise
imd_inscope["msa_name"] = imd_inscope["ca_name"]

# ── 5. Aggregate to MSA level ─────────────────────────────────────────────
print("\n── 5. Aggregating to MSA level ──────────────────────────────────")

MSA_TIERS = {
    "Greater Manchester":            "Established",
    "South Yorkshire":               "Established",
    "West Yorkshire":                "Established",
    "Liverpool City Region":         "Established",
    "Tees Valley":                   "Existing",
    "West Midlands":                 "Established",
    "Cambridgeshire and Peterborough":"Existing",
    "West of England":               "Established",
    "York and North Yorkshire":      "Existing",
    "East Midlands":                 "Existing",
    "North East":                    "Established",
    "Hull and East Yorkshire":       "Existing",
    "Greater Lincolnshire":          "Existing",
    "Lancashire":                    "Established",
    "Sussex & Brighton":             "DPP",
    "Hampshire & Solent":            "DPP",
    "Cheshire & Warrington":         "DPP",
    "Cumbria":                       "DPP",
    "Greater Essex":                 "DPP",
    "Norfolk & Suffolk":             "DPP",
}

rows = []
for msa, grp in imd_inscope.groupby("msa_name"):
    n_lsoas = len(grp)
    if n_lsoas == 0:
        continue

    row = {
        "msa_name":         msa,
        "tier":             MSA_TIERS.get(msa, "Unknown"),
        "n_lsoas":          n_lsoas,
        "imd_avg_rank":     grp["imd_rank"].mean().round(0),
        "imd_median_rank":  grp["imd_rank"].median().round(0),
    }

    # Decile distribution — % of LSOAs in each decile
    for d in range(1, 11):
        row[f"pct_decile_{d}"] = round((grp["imd_decile"] == d).sum() / n_lsoas * 100, 2)

    # Key summary metrics
    row["pct_decile_1"]    = row["pct_decile_1"]    # Most deprived 10%
    row["pct_decile_1_2"]  = round(row["pct_decile_1"] + row["pct_decile_2"], 2)  # Most deprived 20%
    row["pct_decile_9_10"] = round(row["pct_decile_9"] + row["pct_decile_10"], 2) # Least deprived 20%

    # Absolute counts
    row["n_lsoas_decile1"] = (grp["imd_decile"] == 1).sum()
    row["n_lsoas_decile2"] = (grp["imd_decile"] == 2).sum()

    rows.append(row)

msa_df = pd.DataFrame(rows)

# ── 6. Rankings ───────────────────────────────────────────────────────────
print("── 6. Computing rankings ────────────────────────────────────────")

# Higher % in decile 1 = more deprived = rank 1
msa_df["rank_pct_decile1"]   = msa_df["pct_decile_1"].rank(ascending=False, method="min").astype(int)
# Lower avg rank number = more deprived = rank 1
msa_df["rank_avg_rank"]      = msa_df["imd_avg_rank"].rank(ascending=True,  method="min").astype(int)
# Lower median rank = more deprived
msa_df["rank_median_rank"]   = msa_df["imd_median_rank"].rank(ascending=True, method="min").astype(int)

# ── 7. England benchmarks ─────────────────────────────────────────────────
eng_pct_decile1 = 10.0  # By definition — exactly 10% of all LSOAs are in decile 1
eng_avg_rank    = imd["imd_rank"].mean().round(0)
eng_median_rank = imd["imd_rank"].median().round(0)

# ── 8. Print results ──────────────────────────────────────────────────────
print("\n── IMD Results (sorted by % in most deprived 10%) ──────────────")
print(f"\n{'MSA':<36} {'Tier':<12} {'LSOAs':>7} {'%Dec1':>7} {'Rank':>5} {'%Dec1+2':>9} {'AvgRank':>9} {'MedRank':>9}")
print("-" * 100)

for _, r in msa_df.sort_values("pct_decile_1", ascending=False).iterrows():
    print(
        f"{r['msa_name']:<36} "
        f"{r['tier']:<12} "
        f"{r['n_lsoas']:>7,.0f} "
        f"{r['pct_decile_1']:>7.1f}% "
        f"{r['rank_pct_decile1']:>5} "
        f"{r['pct_decile_1_2']:>8.1f}% "
        f"{r['imd_avg_rank']:>9,.0f} "
        f"{r['imd_median_rank']:>9,.0f}"
    )

print(f"\n{'ENGLAND (benchmark)':<36} {'':12} {33755:>7,} {eng_pct_decile1:>7.1f}% {'—':>5} {'20.0%':>9} {eng_avg_rank:>9,.0f} {eng_median_rank:>9,.0f}")

# ── 9. Print YNYCA decile distribution ────────────────────────────────────
print("\n── YNYCA decile distribution ────────────────────────────────────")
ynyca_row = msa_df[msa_df["msa_name"].str.contains("York", case=False)]
if not ynyca_row.empty:
    y = ynyca_row.iloc[0]
    print(f"\n   {'Decile':<10} {'% of LSOAs':>12}  {'Interpretation'}")
    print(f"   {'-'*55}")
    for d in range(1, 11):
        pct  = y[f"pct_decile_{d}"]
        note = " ← most deprived 10% nationally" if d == 1 else (" ← least deprived 10% nationally" if d == 10 else "")
        bar  = "█" * int(pct / 2)
        print(f"   Decile {d:<4}   {pct:>8.1f}%   {bar}{note}")
    print(f"\n   % in most deprived 10%:  {y['pct_decile_1']:.1f}%  (rank {int(y['rank_pct_decile1'])}/20)")
    print(f"   % in most deprived 20%:  {y['pct_decile_1_2']:.1f}%  ")
    print(f"   % in least deprived 20%: {y['pct_decile_9_10']:.1f}%")
    print(f"   Average IMD rank:        {y['imd_avg_rank']:,.0f}  (England avg: {eng_avg_rank:,.0f})")
    print(f"   Median IMD rank:         {y['imd_median_rank']:,.0f}  (England median: {eng_median_rank:,.0f})")

# ── 10. Save ──────────────────────────────────────────────────────────────
out = f"{BASE}/data/processed/msa_imd_ranks.csv"
msa_df.to_csv(out, index=False)
print(f"\n── Saved: {out}")
print("   Next: run update_excel_imd.py to write into tracker")

── 1. Loading IMD 2025 LSOA data ────────────────────────────────
   LSOAs loaded: 33,755
   Decile distribution (England):
imd_decile
1     3375
2     3376
3     3375
4     3376
5     3375
6     3376
7     3375
8     3376
9     3375
10    3376

── 2. Loading File 10 LAD summary ───────────────────────────────
   LADs in File 10: 296
   Columns available: ['lad24_code', 'lad24_name', 'imd_avg_rank', 'imd_rank_of_avg_rank', 'IMD - Average score ', 'IMD - Rank of average score ', 'imd_pct_decile1_file10', 'IMD - Rank of proportion of LSOAs in most deprived 10% nationally ', 'IMD25 - Extent ', 'IMD25 - Rank of extent ', 'IMD25 - Local concentration ', 'IMD25 - Rank of local concentration ']

── 3. Building LSOA → MSA lookup ───────────────────────────────
   North Yorkshire in CA lookup: [['E06000065', 'York and North Yorkshire']]
   LSOAs matched to a CA: 13,327
   LSOAs outside any CA:  20,428

── 4. Adding pipeline MSA areas ─────────────────────────────────
   Sussex & Brighton: 1029 

In [4]:
"""
update_excel_imd.py
===================
Writes IMD 2025 ranks and values into the MSA Indicator Tracker.
Fixes YNYCA name mismatch before writing.

Run from project root:
  python scripts/update_excel_imd.py
"""

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

BASE      = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"
XLSX_PATH = f"{BASE}/data/analysis/MSA_Indicator_Tracker.xlsx"
CSV_PATH  = f"{BASE}/data/processed/msa_imd_ranks.csv"

# ── Load and inspect ───────────────────────────────────────────────────────
ranks = pd.read_csv(CSV_PATH)

print("── MSA names in CSV ─────────────────────────────────────────────")
for n in sorted(ranks["msa_name"].tolist()):
    print(f"   '{n}'")

# ── Fix: confirm correct YNYCA row ────────────────────────────────────────
# Name comes from CA lookup: "York and North Yorkshire"
# Hull row: "Hull and East Yorkshire"
print("\n── Spot check YNYCA vs Hull ──────────────────────────────────────")
for name in ["York and North Yorkshire", "Hull and East Yorkshire"]:
    row = ranks[ranks["msa_name"] == name]
    if not row.empty:
        r = row.iloc[0]
        print(f"\n{name}")
        print(f"  LSOAs:      {r['n_lsoas']}")
        print(f"  % decile 1: {r['pct_decile_1']:.1f}%")
        print(f"  Avg rank:   {r['imd_avg_rank']:,.0f}")
        print(f"  Rank/20:    {r['rank_pct_decile1']}")

# Pull YNYCA correctly
ynyca = ranks[ranks["msa_name"] == "York and North Yorkshire"].iloc[0]

print(f"\n── YNYCA confirmed ──────────────────────────────────────────────")
print(f"  LSOAs:                    {ynyca['n_lsoas']}")
print(f"  % in most deprived 10%:   {ynyca['pct_decile_1']:.1f}%")
print(f"  % in most deprived 20%:   {ynyca['pct_decile_1_2']:.1f}%")
print(f"  % in least deprived 20%:  {ynyca['pct_decile_9_10']:.1f}%")
print(f"  Average IMD rank:         {ynyca['imd_avg_rank']:,.0f}")
print(f"  Median IMD rank:          {ynyca['imd_median_rank']:,.0f}")
print(f"  Rank (% decile 1):        {int(ynyca['rank_pct_decile1'])}/20")

# England benchmarks
ENG_PCT_DEC1   = 10.0
ENG_AVG_RANK   = 16878
ENG_MED_RANK   = 16878

# ── Load workbook ──────────────────────────────────────────────────────────
wb  = load_workbook(XLSX_PATH)
ws2 = wb["2. YNYCA Profile"]
ws3 = wb["3. MSA Comparison"]

GREEN_BG  = "C8E6C9"
AMBER_BG  = "FFF9C4"
DARK      = "1A1A1A"

def write(ws, row, col, value, fmt=None, align="right", bold=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font = Font(name="Arial", size=9, bold=bold, color=DARK)
    c.alignment = Alignment(horizontal=align, vertical="center")
    if fmt:
        c.number_format = fmt
    return c

def mark_status(ws, row, status="Populated"):
    bg = {"Populated": "C8E6C9", "Partial": "FFF9C4", "To do": "FFCCBC"}.get(status, "FFFFFF")
    c = ws.cell(row=row, column=14, value=status)
    c.fill = PatternFill("solid", fgColor=bg)
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="center", vertical="center")

# ── Find rows in Sheet 2 ───────────────────────────────────────────────────
id_to_row = {}
for row in ws2.iter_rows(min_row=4):
    v = row[0].value
    if v:
        id_to_row[v] = row[0].row

# ── Update IMD_001: % LSOAs in most deprived 10% (primary metric) ─────────
r = id_to_row.get("IMD_001")
if r:
    write(ws2, r, 4,  round(ynyca["pct_decile_1"], 1),   fmt="0.0%")  # note: store as decimal
    ws2.cell(row=r, column=4).value = ynyca["pct_decile_1"]           # store as plain number
    ws2.cell(row=r, column=4).number_format = '0.0"%"'
    write(ws2, r, 5,  "% LSOAs",           align="left")
    write(ws2, r, 6,  "2025",              align="left")
    write(ws2, r, 7,  ENG_PCT_DEC1,        fmt='0.0"%"')
    ws2.cell(row=r, column=8).value = f"=D{r}-G{r}"
    ws2.cell(row=r, column=8).number_format = '0.0"%"'
    ws2.cell(row=r, column=8).font = Font(name="Arial", size=9)
    ws2.cell(row=r, column=8).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r, 9,  "Higher = more deprived", align="left")
    write(ws2, r, 13, int(ynyca["rank_pct_decile1"]),
          align="center", bold=True)
    write(ws2, r, 15,
          f"IMD 2025. % of {int(ynyca['n_lsoas'])} LSOAs in YNYCA falling in most deprived 10% nationally. "
          f"England benchmark = 10.0% by definition. "
          f"YNYCA rank {int(ynyca['rank_pct_decile1'])}/20 — least deprived MSA. "
          f"Decile distribution heavily skewed toward less deprived end (median rank {ynyca['imd_median_rank']:,.0f} vs England {ENG_MED_RANK:,}).",
          align="left")
    mark_status(ws2, r, "Populated")
    print(f"  ✓  IMD_001 updated — row {r}")

# ── Update IMD_002: % LSOAs in most deprived 20% ──────────────────────────
r2 = id_to_row.get("IMD_002")
if r2:
    write(ws2, r2, 4,  round(ynyca["pct_decile_1_2"], 1))
    write(ws2, r2, 5,  "% LSOAs",           align="left")
    write(ws2, r2, 6,  "2025",              align="left")
    write(ws2, r2, 7,  20.0)
    ws2.cell(row=r2, column=8).value = f"=D{r2}-G{r2}"
    ws2.cell(row=r2, column=8).font = Font(name="Arial", size=9)
    ws2.cell(row=r2, column=8).alignment = Alignment(horizontal="right", vertical="center")
    write(ws2, r2, 9,  "Higher = more deprived", align="left")
    write(ws2, r2, 15,
          f"IMD 2025. % of LSOAs in deciles 1 or 2 (most deprived 20% nationally). "
          f"England benchmark = 20.0% by definition. "
          f"YNYCA {ynyca['pct_decile_1_2']:.1f}% — well below benchmark, consistent with rural affluent character. "
          f"Avg IMD rank {ynyca['imd_avg_rank']:,.0f} vs England avg {ENG_AVG_RANK:,}.",
          align="left")
    mark_status(ws2, r2, "Populated")
    print(f"  ✓  IMD_002 updated — row {r2}")

# ── Update Sheet 3: all 20 MSAs ────────────────────────────────────────────
print("\n── Updating Sheet 3 MSA Comparison ─────────────────────────────")

# Name map: Sheet 3 header name → CSV msa_name
NAME_MAP = {
    "Greater Manchester":            "Greater Manchester",
    "West Midlands":                 "West Midlands",
    "North East":                    "North East",
    "Liverpool City Region":         "Liverpool City Region",
    "South Yorkshire":               "South Yorkshire",
    "West Yorkshire":                "West Yorkshire",
    "West of England":               "West of England",
    "Lancashire":                    "Lancashire",
    "Tees Valley":                   "Tees Valley",
    "York & North Yorkshire":        "York and North Yorkshire",
    "Hull & East Yorkshire":         "Hull and East Yorkshire",
    "Greater Lincolnshire":          "Greater Lincolnshire",
    "East Midlands":                 "East Midlands",
    "Cambridgeshire & Pboro":        "Cambridgeshire and Peterborough",
    "Sussex & Brighton":             "Sussex & Brighton",
    "Hampshire & Solent":            "Hampshire & Solent",
    "Cheshire & Warrington":         "Cheshire & Warrington",
    "Cumbria":                       "Cumbria",
    "Greater Essex":                 "Greater Essex",
    "Norfolk & Suffolk":             "Norfolk & Suffolk",
}

# Find column positions in Sheet 3 row 2
msa_cols = {}
for col in range(1, 30):
    v = ws3.cell(row=2, column=col).value
    if v:
        parts = str(v).split("\n")
        name = parts[1].strip() if len(parts) >= 2 else parts[0].strip()
        msa_cols[name] = col

# Find IMD indicator rows in Sheet 3 column A
comp_id_rows = {}
for row in ws3.iter_rows(min_row=3):
    v = row[0].value
    if v:
        comp_id_rows[v] = row[0].row

imd001_row = comp_id_rows.get("IMD_001")
imd002_row = comp_id_rows.get("IMD_002")

if not imd001_row:
    # IMD_001 not yet in Sheet 3 — find next empty row and add it
    max_row = max(comp_id_rows.values()) if comp_id_rows else 3
    imd001_row = max_row + 1
    ws3.cell(row=imd001_row, column=1).value = "IMD_001"
    ws3.cell(row=imd001_row, column=2).value = "% LSOAs in most deprived 10%"
    ws3.cell(row=imd001_row, column=3).value = "%"
    ws3.cell(row=imd001_row, column=4).value = ENG_PCT_DEC1
    print(f"  Added IMD_001 to Sheet 3 row {imd001_row}")

if not imd002_row:
    imd002_row = imd001_row + 1
    ws3.cell(row=imd002_row, column=1).value = "IMD_002"
    ws3.cell(row=imd002_row, column=2).value = "% LSOAs in most deprived 20%"
    ws3.cell(row=imd002_row, column=3).value = "%"
    ws3.cell(row=imd002_row, column=4).value = 20.0
    print(f"  Added IMD_002 to Sheet 3 row {imd002_row}")

for sheet_name, csv_name in NAME_MAP.items():
    col = msa_cols.get(sheet_name)
    if col is None:
        print(f"  ⚠  Column not found: '{sheet_name}'")
        continue

    row_data = ranks[ranks["msa_name"] == csv_name]
    if row_data.empty:
        print(f"  ⚠  No data for: '{csv_name}'")
        continue

    r = row_data.iloc[0]

    if imd001_row:
        c = ws3.cell(row=imd001_row, column=col,
                     value=round(float(r["pct_decile_1"]), 1))
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")
        # Highlight rank 1-5 red, rank 16-20 green
        rank = int(r["rank_pct_decile1"])
        if rank <= 5:
            c.fill = PatternFill("solid", fgColor="FFCDD2")
        elif rank >= 16:
            c.fill = PatternFill("solid", fgColor="C8E6C9")

    if imd002_row:
        c = ws3.cell(row=imd002_row, column=col,
                     value=round(float(r["pct_decile_1_2"]), 1))
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")

    print(f"  ✓  {sheet_name:<32}  %dec1={r['pct_decile_1']:.1f}%  rank={int(r['rank_pct_decile1'])}")

# ── Save ───────────────────────────────────────────────────────────────────
wb.save(XLSX_PATH)
print(f"\n── Saved: {XLSX_PATH}")

# ── Print summary for YNYCA profile ───────────────────────────────────────
print("\n── YNYCA IMD summary for profile ────────────────────────────────")
print(f"  % LSOAs in most deprived 10%:   {ynyca['pct_decile_1']:.1f}%  (rank 20/20 — least deprived MSA)")
print(f"  % LSOAs in most deprived 20%:   {ynyca['pct_decile_1_2']:.1f}%  (England benchmark: 20.0%)")
print(f"  % LSOAs in least deprived 20%:  {ynyca['pct_decile_9_10']:.1f}%")
print(f"  Average IMD rank:               {ynyca['imd_avg_rank']:,.0f}  (England: {ENG_AVG_RANK:,})")
print(f"  Rank among 20 MSAs:             {int(ynyca['rank_pct_decile1'])}/20")
print(f"\n  Decile distribution:")
for d in range(1, 11):
    pct = ynyca[f"pct_decile_{d}"]
    bar = "█" * int(pct / 2)
    flag = " ← most deprived" if d == 1 else (" ← least deprived" if d == 10 else "")
    print(f"  Decile {d:<3}  {pct:>5.1f}%  {bar}{flag}")

── MSA names in CSV ─────────────────────────────────────────────
   'Cambridgeshire and Peterborough'
   'Cheshire & Warrington'
   'Cumbria'
   'East Midlands'
   'Greater Essex'
   'Greater Lincolnshire'
   'Greater Manchester'
   'Hampshire & Solent'
   'Hull and East Yorkshire'
   'Lancashire'
   'Liverpool City Region'
   'Norfolk & Suffolk'
   'North East'
   'South Yorkshire'
   'Sussex & Brighton'
   'Tees Valley'
   'West Midlands'
   'West Yorkshire'
   'West of England'
   'York and North Yorkshire'

── Spot check YNYCA vs Hull ──────────────────────────────────────

York and North Yorkshire
  LSOAs:      500
  % decile 1: 2.0%
  Avg rank:   21,081
  Rank/20:    20

Hull and East Yorkshire
  LSOAs:      381
  % decile 1: 21.8%
  Avg rank:   15,867
  Rank/20:    6

── YNYCA confirmed ──────────────────────────────────────────────
  LSOAs:                    500
  % in most deprived 10%:   2.0%
  % in most deprived 20%:   6.2%
  % in least deprived 20%:  28.6%
  Average IMD r

In [5]:
print(msa_df[["msa_name","imd_avg_rank","rank_avg_rank",
              "imd_median_rank","rank_median_rank"]]
      .sort_values("imd_avg_rank").to_string())

                           msa_name  imd_avg_rank  rank_avg_rank  imd_median_rank  rank_median_rank
16                    West Midlands       11698.0              1           8361.0                 1
13                  South Yorkshire       12398.0              2           9980.0                 2
10            Liverpool City Region       12509.0              3           9994.0                 3
15                      Tees Valley       13001.0              4          10413.0                 4
12                       North East       13375.0              5          10928.0                 5
6                Greater Manchester       13461.0              6          11140.0                 6
17                   West Yorkshire       14013.0              7          12158.0                 7
9                        Lancashire       14586.0              8          13758.0                 8
5              Greater Lincolnshire       15439.0              9          15475.0                 9


In [ ]:
import pandas as pd, altair as alt, numpy as np, warnings
warnings.filterwarnings('ignore')

BASE = '/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map'

f7 = pd.read_csv(f'{BASE}/data/raw/imd/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv')
f7 = f7.rename(columns={
    'LSOA code (2021)': 'lsoa_code',
    'Local Authority District code (2024)': 'lad24_code',
    'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)': 'imd_d',
    'Income Decile (where 1 is most deprived 10% of LSOAs)': 'income_d',
    'Employment Decile (where 1 is most deprived 10% of LSOAs)': 'employ_d',
    'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)': 'education_d',
    'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)': 'health_d',
    'Crime Decile (where 1 is most deprived 10% of LSOAs)': 'crime_d',
    'Barriers to Housing and Services Decile (where 1 is most deprived 10% of LSOAs)': 'barriers_d',
    'Living Environment Decile (where 1 is most deprived 10% of LSOAs)': 'living_d',
    'Geographical Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'geo_barriers_d',
    'Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'indoors_d',
})
ca  = pd.read_csv(f'{BASE}/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv')
dpp = pd.read_csv(f'{BASE}/data/raw/lookups/dpp_lad_lookup_manual.csv')
lad_msa = pd.concat([
    ca[['LAD25CD','CAUTH25NM']].rename(columns={'LAD25CD':'lad24_code','CAUTH25NM':'msa_name'}),
    dpp[['lad25_code','dpp_area']].rename(columns={'lad25_code':'lad24_code','dpp_area':'msa_name'}),
]).drop_duplicates('lad24_code')
f7 = f7.merge(lad_msa, on='lad24_code', how='left')
msa_all = f7[f7['msa_name'].notna()].copy()

DCOLS = ['imd_d','income_d','employ_d','education_d','health_d',
         'crime_d','barriers_d','living_d','geo_barriers_d','indoors_d']
msa_avg = msa_all.groupby('msa_name')[DCOLS].mean().reset_index()
for c in DCOLS:
    msa_avg[f'rank_{c}'] = msa_avg[c].rank(ascending=False).astype(int)

N = len(msa_avg)
YNYCA_LADS = ['E06000014','E06000065']
ynyca     = f7[f7['lad24_code'].isin(YNYCA_LADS)].copy()
ynyca_row = msa_avg[msa_avg['msa_name'] == 'York and North Yorkshire'].iloc[0]

# ── Color helpers ─────────────────────────────────────────────────────────────
C_TEAL  = '#36b7b4'
C_AMBER = '#e8a84a'
C_RED   = '#e6224b'

def rank_color(r):
    if r <= 10: return C_TEAL
    if r <= 14: return C_AMBER
    return C_RED

def pct_color(p):
    if p <= 20: return C_TEAL
    if p <= 40: return C_AMBER
    return C_RED

def rank_opacity(r):
    if r <= 10:  return 0.20 + 0.25 * ((r - 1) / 9)
    elif r <= 14: return 0.30 + 0.15 * ((r - 11) / 3)
    else:         return 0.40 + 0.25 * ((r - 15) / 6)

def pct_opacity(p):
    return float(np.clip(0.15 + 0.50 * (p / 100), 0.15, 0.65))

# ── Layout ────────────────────────────────────────────────────────────────────
W     = 620
ROW_H = 30
PAD_V = 6

RANK_X1, RANK_X2 = 248, 358
PCT_X1,  PCT_X2  = 374, 606

DX_DOMAIN = 12 - W/2
DX_RANK   = (RANK_X1 + RANK_X2)/2 - W/2
DX_PCT    = (PCT_X1  + PCT_X2) /2 - W/2
DX_META   = (RANK_X1 + PCT_X2) /2 - W/2

# ── Domain metadata: sort_order descending → Composite at top, Indoors at bottom
# Use natural IMD order: Composite, Income, Employ, Education, Health, Crime,
# Barriers, Living Environment, then sub-domains (Geog Barriers, Indoors)
DOMAIN_META = [
    ('imd_d',          'Composite IMD',        False, 10),
    ('income_d',       'Income',               False, 9),
    ('employ_d',       'Employment',           False, 8),
    ('education_d',    'Education & Skills',   False, 7),
    ('health_d',       'Health & Disability',  False, 6),
    ('crime_d',        'Crime',                False, 5),
    ('barriers_d',     'Barriers to Services', False, 4),
    ('living_d',       'Living Environment',   False, 3),
    ('geo_barriers_d', 'Geog. Barriers',       True,  2),
    ('indoors_d',      'Indoors Living',       True,  1),
]

rows = []
for col, label, is_sub, order in DOMAIN_META:
    rank = int(ynyca_row[f'rank_{col}'])
    pct  = float((ynyca[col] <= 2).mean() * 100)
    rows.append({
        'domain':       label,
        'is_sub':       is_sub,
        'rank_label':   str(rank),
        'pct_label':    f'{round(pct):.0f}%',
        'rank_color':   rank_color(rank),
        'pct_color':    pct_color(pct),
        'rank_opacity': float(rank_opacity(rank)),
        'pct_opacity':  pct_opacity(pct),
        'sort_order':   order,
        'row_type':     'data',
        'alt_bg':       order % 2 == 1,  # odd sort_orders: Income(9), Education(7), Crime(5), Living(3), Indoors(1)
        'rank_x1': float(RANK_X1), 'rank_x2': float(RANK_X2),
        'pct_x1':  float(PCT_X1),  'pct_x2':  float(PCT_X2),
    })

# Column-label header — unique domain name so it gets its own y slot
rows.append({
    'domain': '__COL_HDR__', 'is_sub': False,
    'rank_label': '', 'pct_label': '',
    'rank_color': '#aaa', 'pct_color': '#aaa',
    'rank_opacity': 0.0, 'pct_opacity': 0.0,
    'sort_order': 12, 'row_type': 'col_hdr', 'alt_bg': False,
    'rank_x1': float(RANK_X1), 'rank_x2': float(RANK_X2),
    'pct_x1':  float(PCT_X1),  'pct_x2':  float(PCT_X2),
})

# Meta-header — unique domain name
rows.append({
    'domain': '__META_HDR__', 'is_sub': False,
    'rank_label': '', 'pct_label': '',
    'rank_color': '#aaa', 'pct_color': '#aaa',
    'rank_opacity': 0.0, 'pct_opacity': 0.0,
    'sort_order': 13, 'row_type': 'meta_hdr', 'alt_bg': False,
    'rank_x1': float(RANK_X1), 'rank_x2': float(RANK_X2),
    'pct_x1':  float(PCT_X1),  'pct_x2':  float(PCT_X2),
})

df = pd.DataFrame(rows)
# Descending: 13→12→10→9→…→1  ⟹  meta at top, col_hdr, then domains, Indoors at bottom
DOM_ORDER = df.sort_values('sort_order', ascending=False)['domain'].tolist()

Y       = alt.Y('domain:N', sort=DOM_ORDER,
                axis=alt.Axis(labels=False, ticks=False, domain=False, title=None))
X_SCALE = alt.Scale(domain=[0, W])

data_df     = df[df['row_type'] == 'data']
main_df     = data_df[~data_df['is_sub']]
sub_df      = data_df[data_df['is_sub']]
col_hdr_df  = df[df['row_type'] == 'col_hdr']
meta_hdr_df = df[df['row_type'] == 'meta_hdr']

# ── Row backgrounds ───────────────────────────────────────────────────────────
alt_bg = alt.Chart(df[df['alt_bg']]).mark_rect(color='#f6f6f6').encode(
    y=Y, x=alt.value(0), x2=alt.value(W))

col_hdr_bg = alt.Chart(col_hdr_df).mark_rect(color='#f0f0f0').encode(
    y=Y, x=alt.value(0), x2=alt.value(W))

meta_hdr_bg = alt.Chart(meta_hdr_df).mark_rect(color='#f0f0f0').encode(
    y=Y, x=alt.value(0), x2=alt.value(W))

# ── Colored cells ─────────────────────────────────────────────────────────────
rank_cells = alt.Chart(data_df).mark_rect(
    height=ROW_H - PAD_V, cornerRadius=4,
).encode(
    y=Y,
    x=alt.X('rank_x1:Q', scale=X_SCALE, axis=None),
    x2='rank_x2:Q',
    color=alt.Color('rank_color:N', scale=None, legend=None),
    opacity=alt.Opacity('rank_opacity:Q', scale=alt.Scale(domain=[0,1], range=[0,1]), legend=None),
)

pct_cells = alt.Chart(data_df).mark_rect(
    height=ROW_H - PAD_V, cornerRadius=4,
).encode(
    y=Y,
    x=alt.X('pct_x1:Q', scale=X_SCALE, axis=None),
    x2='pct_x2:Q',
    color=alt.Color('pct_color:N', scale=None, legend=None),
    opacity=alt.Opacity('pct_opacity:Q', scale=alt.Scale(domain=[0,1], range=[0,1]), legend=None),
)

# ── Domain text ───────────────────────────────────────────────────────────────
txt_main = alt.Chart(main_df).mark_text(
    align='left', fontSize=12, fontWeight='bold', color='#111111', dx=DX_DOMAIN,
).encode(y=Y, text='domain:N')

txt_sub = alt.Chart(sub_df).mark_text(
    align='left', fontSize=11.5, fontStyle='italic', color='#333333', dx=DX_DOMAIN + 12,
).encode(y=Y, text='domain:N')

# ── Column headers ────────────────────────────────────────────────────────────
hdr_domain = alt.Chart(col_hdr_df).mark_text(
    align='left', fontSize=9, fontWeight='bold', color='#888888', dx=DX_DOMAIN,
).encode(y=Y, text=alt.value('DOMAIN'))

hdr_rank = alt.Chart(col_hdr_df).mark_text(
    align='center', fontSize=9, fontWeight='bold', color='#888888', dx=DX_RANK,
).encode(y=Y, text=alt.value('RANK / 21'))

hdr_pct = alt.Chart(col_hdr_df).mark_text(
    align='center', fontSize=9, fontWeight='bold', color='#888888', dx=DX_PCT,
).encode(y=Y, text=alt.value('% OF LSOAS IN WORST QUINTILE'))

# ── Meta-header ───────────────────────────────────────────────────────────────
hdr_meta = alt.Chart(meta_hdr_df).mark_text(
    align='center', fontSize=9, fontWeight='bold', color='#888888', dx=DX_META,
).encode(y=Y, text=alt.value('DEPRIVATION PROFILE'))

# ── Value text ────────────────────────────────────────────────────────────────
txt_rank = alt.Chart(data_df).mark_text(
    align='center', fontSize=13, fontWeight='bold', color='#111111', dx=DX_RANK,
).encode(y=Y, text='rank_label:N')

txt_pct = alt.Chart(data_df).mark_text(
    align='center', fontSize=12, color='#111111', dx=DX_PCT,
).encode(y=Y, text='pct_label:N')

# ── Footnote ──────────────────────────────────────────────────────────────────
N_ROWS = len(DOMAIN_META) + 2  # data rows + 2 header rows
H = ROW_H * N_ROWS

chart_domain_table = (
    alt_bg + col_hdr_bg + meta_hdr_bg
    + rank_cells + pct_cells
    + txt_main + txt_sub
    + hdr_domain + hdr_rank + hdr_pct + hdr_meta
    + txt_rank + txt_pct
).properties(
    width=W, height=H,
).configure_view(strokeWidth=0).configure(background='#ffffff').configure_axis(grid=False)

out = f'{BASE}/data/processed/charts/ynyca_domain_ranks_table.png'
chart_domain_table.save(out, scale_factor=3)
print(f'Saved {out}')
chart_domain_table
